# DSFB-ATLAS Reproducibility Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/infinityabundance/dsfb/blob/main/crates/dsfb-atlas/colab/dsfb_atlas_reproduce.ipynb)

This notebook reproduces the **10,000-theorem build** of the **DSFB-ATLAS Alternative Deterministic Residual Theorem Atlas (v2.0)** end-to-end in a free Colab runtime. Expected wall time: ~6&ndash;9 minutes on the default Colab CPU.

**DOI:** https://doi.org/10.5281/zenodo.19798649  
**ORCID:** https://orcid.org/0009-0006-1155-027X  
**License:** Apache-2.0 (reference implementation only; the underlying theoretical framework is proprietary Background IP of [Invariant Forge LLC](https://invariantforge.net), licensing@invariantforge.net)  
**Author:** Riaan de Beer &mdash; Chief Research Advisor, Invariant Forge LLC &mdash; riaan@invariantforge.net

**Citation:**
> de Beer, R. (2026). *DSFB-ATLAS Alternative Deterministic Residual Theorem Atlas: A 10,000-Theorem Universality Framework for Operator-Legible Deterministic Residual Inference &mdash; Drift&ndash;Slew, Envelope, Grammar, Trust, and Endoductive Structural Inference.* (v2.0). Zenodo. https://doi.org/10.5281/zenodo.19798649

## 1. Install the Rust toolchain (rustup, stable)

The crate is `edition = "2021"`; stable Rust is sufficient.

In [ ]:
%%bash
set -euo pipefail
curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable --profile minimal
echo 'source "$HOME/.cargo/env"' >> ~/.bashrc
source "$HOME/.cargo/env"
rustc --version
cargo --version

## 2. Clone the dsfb repository

In [ ]:
%%bash
set -euo pipefail
cd /content
rm -rf dsfb
git clone --depth 1 https://github.com/infinityabundance/dsfb.git
cd dsfb
git rev-parse --short HEAD > /content/dsfb_git_hash.txt
echo "git hash: $(cat /content/dsfb_git_hash.txt)"
ls crates/dsfb-atlas
ls crates/dsfb-bank/spec/atlas | head

## 3. Build `dsfb-atlas` in release mode

First build downloads dependencies and takes ~3&ndash;5 minutes; cached rebuilds are ~10 s.

In [ ]:
%%bash
set -euo pipefail
source "$HOME/.cargo/env"
cd /content/dsfb
# `-p dsfb-atlas` selects the package explicitly because the workspace's
# default-members is set to `crates/dsfb`, so `--bin dsfb-atlas` alone
# would search the wrong package.
cargo build --release -p dsfb-atlas
ls -lh target/release/dsfb-atlas

## 4. Run the generator AND compile the 10,000-theorem atlas PDF

The generator produces 10 LaTeX Part files into a private working directory; the notebook then compiles them into a self-contained `dsfb_atlas_10000_theorems.pdf` and surfaces ONLY that PDF (plus the `dedup_report.json` attestation and the structural `coverage_heatmap.png`). The intermediate `.tex` content is not surfaced or bundled. The full typeset 4{,}846-page DSFB-ATLAS paper lives on Zenodo at the DOI cited in the closing cell.

In [ ]:
%%bash
set -euo pipefail
source "$HOME/.cargo/env"
cd /content/dsfb
GIT_HASH=$(cat /content/dsfb_git_hash.txt)

# 4.1  Generate the 10 part_*.tex files into a PRIVATE working dir.
PRIV_OUT=/tmp/dsfb_atlas_priv
rm -rf "$PRIV_OUT" && mkdir -p "$PRIV_OUT"
./target/release/dsfb-atlas \
    --spec-dir crates/dsfb-bank/spec/atlas \
    --bank-spec-dir crates/dsfb-bank/spec \
    --out "$PRIV_OUT" \
    --git-hash "$GIT_HASH" \
    > /tmp/dsfb_atlas_run.log 2>&1
tail -3 /tmp/dsfb_atlas_run.log

# 4.2  Install a minimal TeX Live so we can compile the parts to PDF.
apt-get -qq update >/dev/null
DEBIAN_FRONTEND=noninteractive apt-get -qq install -y --no-install-recommends \
    texlive-latex-base texlive-latex-extra texlive-latex-recommended \
    texlive-fonts-recommended >/dev/null
echo 'TeX Live ready'

# 4.3  Stage the minimal preamble + write a thin master that \include's the parts.
BUILD=/tmp/dsfb_atlas_build
rm -rf "$BUILD" && mkdir -p "$BUILD"
cp crates/dsfb-atlas/colab/atlas_minimal.sty "$BUILD/atlas_minimal.sty"
for i in 01 02 03 04 05 06 07 08 09 10; do
    cp "$PRIV_OUT/part_${i}.tex" "$BUILD/part_${i}.tex"
done
cat > "$BUILD/dsfb_atlas_10000_theorems.tex" <<'EOF'
\documentclass[10pt,oneside,openany]{report}
\usepackage{atlas_minimal}
\title{DSFB-ATLAS \\ The 10{,}000-Theorem Atlas \\ \large Generated artefact (auto-built; structurally unique proof bodies)}
\author{Riaan de Beer, Invariant Forge LLC}
\date{Generated \today}
\begin{document}
\maketitle
\tableofcontents
\include{part_01}
\include{part_02}
\include{part_03}
\include{part_04}
\include{part_05}
\include{part_06}
\include{part_07}
\include{part_08}
\include{part_09}
\include{part_10}
\end{document}
EOF

# 4.4  Compile (2 passes for ToC convergence). pdflatex is ~5x faster than
#      lualatex for this content; expect 2-5 min wall time.
cd "$BUILD"
pdflatex -interaction=nonstopmode -halt-on-error dsfb_atlas_10000_theorems.tex > /tmp/pdf_pass1.log 2>&1 || (tail -20 /tmp/pdf_pass1.log; exit 1)
pdflatex -interaction=nonstopmode -halt-on-error dsfb_atlas_10000_theorems.tex > /tmp/pdf_pass2.log 2>&1 || (tail -20 /tmp/pdf_pass2.log; exit 1)
tail -1 /tmp/pdf_pass2.log
ls -lh "$BUILD/dsfb_atlas_10000_theorems.pdf"

# 4.5  Surface ONLY the public artefacts: PDF + JSON + (heatmap PNG comes later).
PUB_OUT=/content/dsfb_atlas_attestation
rm -rf "$PUB_OUT" && mkdir -p "$PUB_OUT"
cp "$BUILD/dsfb_atlas_10000_theorems.pdf" "$PUB_OUT/dsfb_atlas_10000_theorems.pdf"
cp "$PRIV_OUT/dedup_report.json"          "$PUB_OUT/dedup_report.json"

# 4.6  Wipe the .tex working dirs — no LaTeX content surfaced.
rm -rf "$PRIV_OUT" "$BUILD"
ls -lh "$PUB_OUT"

## 5. Verify the build attestation

The canonical claim is **10,000 emitted, 10,000 unique proof hashes, 0 collisions**. The cell below asserts exactly that triple from the public attestation JSON &mdash; if it fails, the notebook errors visibly.

In [ ]:
import json, pathlib
report = json.loads(pathlib.Path('/content/dsfb_atlas_attestation/dedup_report.json').read_text())
# Print only the attestation triple + git hash (no LaTeX content surfaced).
print(json.dumps({
    'total': report['total'],
    'unique': report['unique'],
    'collisions': report['collisions'],
    'git_hash': report['git_hash'],
}, indent=2))
assert report['total'] == 10000, f"expected 10000 emitted, got {report['total']}"
assert report['unique'] == 10000, f"expected 10000 unique hashes, got {report['unique']}"
assert report['collisions'] == [], f"unexpected collisions: {report['collisions']}"
print('ATTESTATION OK: 10000 / 10000 / 0 collisions')

## 6. Structural coverage from the public YAML

The 10,000-theorem grid is `10 Parts × 10 Chapters × 10 method stems × 10 method modifiers`. The next two cells visualise the structural shape of the atlas using ONLY the public YAML inputs &mdash; no LaTeX content is read or surfaced.

In [ ]:
import yaml, pathlib
spec_dir = pathlib.Path('/content/dsfb/crates/dsfb-bank/spec/atlas')
per_part = {}
for pf in sorted(spec_dir.glob('P*.yaml')):
    spec = yaml.safe_load(pf.read_text())
    n = sum(len(ch['stems']) * len(ch['modifiers']) for ch in spec['chapters'])
    per_part[spec['part_id']] = n
for k, v in per_part.items():
    print(f'  {k}: {v} theorems')
print(f'  TOTAL: {sum(per_part.values())} theorems')
assert sum(per_part.values()) == 10000

## 7. Coverage heatmap (Part × Chapter)

Plot the 10 × 10 grid of theorem counts from the public YAML Part specs. Each cell is `len(stems) * len(modifiers) = 100`, so the heatmap is uniform &mdash; this is a sanity check on the input shape, not a leak of LaTeX content.

In [ ]:
!pip install -q pyyaml matplotlib

import yaml, pathlib, numpy as np, matplotlib.pyplot as plt

spec_dir = pathlib.Path('/content/dsfb/crates/dsfb-bank/spec/atlas')
parts = sorted(p for p in spec_dir.glob('P*.yaml'))
grid = np.zeros((10, 10), dtype=int)
ylabels = []
for i, pf in enumerate(parts):
    spec = yaml.safe_load(pf.read_text())
    ylabels.append(spec['part_id'])
    for j, ch in enumerate(spec['chapters']):
        grid[i, j] = len(ch['stems']) * len(ch['modifiers'])

fig, ax = plt.subplots(figsize=(9, 6))
im = ax.imshow(grid, cmap='viridis', vmin=0, vmax=100)
ax.set_xticks(range(10)); ax.set_xticklabels([f'C{j+1:02d}' for j in range(10)])
ax.set_yticks(range(10)); ax.set_yticklabels(ylabels)
ax.set_title('DSFB-ATLAS coverage: theorems per (Part, Chapter)')
for i in range(10):
    for j in range(10):
        ax.text(j, i, str(grid[i, j]), ha='center', va='center', color='white', fontsize=8)
fig.colorbar(im, ax=ax, label='theorems')
plt.tight_layout()
import pathlib
pathlib.Path('/content/dsfb_atlas_attestation').mkdir(parents=True, exist_ok=True)
plt.savefig('/content/dsfb_atlas_attestation/coverage_heatmap.png', dpi=150)
plt.show()
print(f'Grid sum: {grid.sum()} (expected 10000)')

## 8. Download the public artefact bundle

Bundles **only** the public-facing artefacts: `dsfb_atlas_10000_theorems.pdf` (the typeset 10,000-theorem atlas, generated above), `dedup_report.json` (the SHA-256 attestation), and `coverage_heatmap.png` (the structural-shape PNG). The LaTeX source is *not* included &mdash; for the typeset 4{,}846-page full DSFB-ATLAS paper, see the Zenodo deposit at the DOI in the closing cell.

In [ ]:
import shutil, pathlib
pub_dir = pathlib.Path('/content/dsfb_atlas_attestation')
bundle = pathlib.Path('/content/dsfb_atlas_attestation.zip')
if bundle.exists():
    bundle.unlink()
shutil.make_archive(str(bundle.with_suffix('')), 'zip', root_dir=str(pub_dir))
print(f'Bundle: {bundle} ({bundle.stat().st_size:,} bytes)')
print('Contents:')
for f in sorted(pub_dir.glob('*')):
    print(f'  - {f.name} ({f.stat().st_size:,} bytes)')
try:
    from google.colab import files
    files.download(str(bundle))
except Exception as e:
    print(f'(skip auto-download: {e})')

## Closing

If the assertion in Section 5 succeeded, you have independently reproduced the SHA-256 attestation: 10,000 atlas theorems with 10,000 distinct proof bodies. The typeset 4{,}846-page DSFB-ATLAS PDF lives on Zenodo at the DOI below; the LaTeX source for the paper is not distributed via this notebook.

**DSFB-ATLAS PDF (Zenodo):** https://doi.org/10.5281/zenodo.19798649

**Cite as:**
> de Beer, R. (2026). *DSFB-ATLAS Alternative Deterministic Residual Theorem Atlas: A 10,000-Theorem Universality Framework for Operator-Legible Deterministic Residual Inference &mdash; Drift&ndash;Slew, Envelope, Grammar, Trust, and Endoductive Structural Inference.* (v2.0). Zenodo. https://doi.org/10.5281/zenodo.19798649